# 01 — Data Audit and Cleaning
## Iowa Liquor Sales, January–July 2026

**Goal:** create a reproducible, analysis-ready version of the 2026 Iowa Liquor Sales snapshot before doing business analysis or modeling.

This notebook answers five questions:

1. Do the downloaded CSV source file(s) have a consistent schema and non-overlapping records?
2. What is missing, duplicated, malformed, or logically inconsistent?
3. Which unusual values are legitimate business events, such as returns?
4. Can source quality issues be repaired without inventing data?
5. What fields should downstream notebooks trust?

### Verified project snapshot

The snapshot used to develop this project was downloaded from the official Iowa Data Hub and covers **January 1 through July 31, 2026**. The portal export arrived as three CSV files, but the notebook does **not** depend on that exact split or those filenames.

- 1,402,652 raw rows
- 23 source columns
- Date range: 2026-01-01 through 2026-07-31
- 13 extra exact duplicate rows
- 1,088 negative sales rows, overwhelmingly return invoices
- 680 rows initially missing store geography
- 25 rows initially missing category information
- A January 2026 precision issue affects several source numeric fields

The cleaning philosophy is deliberately conservative: **preserve real transactions, repair only what can be derived from the data or supported by traceable evidence, and flag uncertainty instead of silently deleting it.**


## 1. Setup and data requirements

### Data source

This project uses the public **Iowa Liquor Sales, 2026** dataset published by the Iowa Department of Revenue on the Iowa Data Hub:

https://data.iowa.gov/catalog/dataset/1263

The raw data files are **not stored in this GitHub repository** because the dataset is large. To reproduce the analysis:

1. Open the Iowa Liquor Sales, 2026 dataset above and download the data as CSV. The portal may provide one CSV or split a large export into multiple CSV files.
2. Place **all CSV files from that export** in `data/raw/`.
3. Do not rename or manually edit the raw files. The notebook accepts one or more `.csv` files and does not depend on the original three-part export filenames.
4. Run this notebook. The analysis is explicitly limited to **2026-01-01 through 2026-07-31**, so a later 2026 download can still reproduce the intended analysis window.

Expected project structure:

```text
Iowa-Liquor-Sales/
├── data/
│   ├── raw/
│   │   └── <one or more Iowa Liquor Sales CSV files>
│   └── processed/
└── notebooks/
    └── 01_data_audit_cleaning.ipynb
```

Keep the raw files unchanged. All corrections and derived fields will be written to `data/processed/`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# Reproducibility settings
DATASET_URL = "https://data.iowa.gov/catalog/dataset/1263"
ANALYSIS_START = pd.Timestamp("2026-01-01")
ANALYSIS_END = pd.Timestamp("2026-07-31")

# Works whether VS Code opens the project root or the notebooks folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Create the expected folders if they do not already exist.
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Accept one or more CSV files; the analysis should not depend on how the portal split the export.
FILES = sorted(RAW_DIR.glob("*.csv"))

if not FILES:
    raise FileNotFoundError(
        "No CSV files were found in the raw-data folder:\n"
        f"{RAW_DIR}\n\n"
        "Download the Iowa Liquor Sales, 2026 dataset as CSV from:\n"
        f"{DATASET_URL}\n\n"
        "Place all downloaded CSV file(s) in data/raw/ and rerun this cell."
    )

print(f"Found {len(FILES)} raw CSV file(s):")
for path in FILES:
    print(f"  - {path.name}")


## 2. Validate the source files before concatenating

A large export may arrive as one CSV or several chunks. Multiple files can fail in subtle ways: one file may have a different column order, a different schema, overlapping records, or an unexpected date range. We validate every source file before treating them as one table.


In [ ]:
headers = {p.name: pd.read_csv(p, nrows=0).columns.tolist() for p in FILES}
reference_header = headers[FILES[0].name]

schema_check = pd.DataFrame({
    "file": [p.name for p in FILES],
    "column_count": [len(headers[p.name]) for p in FILES],
    "schema_matches_first_file": [headers[p.name] == reference_header for p in FILES],
})

schema_check

In [ ]:
reference_header

### Identifier columns must stay as text

Fields such as `store_no`, `item_no`, ZIP code, and FIPS code are identifiers, not quantities. Reading them as numbers can destroy leading zeros. For example, this snapshot contains store IDs such as `010206`.

In [ ]:
ID_DTYPES = {
    "invoice_id": "string",
    "store_no": "string",
    "store_zip_code": "string",
    "county_fips_code": "string",
    "category_code": "string",
    "vendor_number": "string",
    "item_no": "string",
}

frames = []
file_summary = []

for path in FILES:
    part = pd.read_csv(
        path,
        dtype=ID_DTYPES,
        parse_dates=["ordered_on"],
        low_memory=False,
    )
    # Restrict the project to the fixed Jan-Jul 2026 analysis window.
    part = part.loc[part["ordered_on"].between(ANALYSIS_START, ANALYSIS_END)].copy()

    part["_source_file"] = path.name
    frames.append(part)
    file_summary.append({
        "file": path.name,
        "rows": len(part),
        "date_min": part["ordered_on"].min(),
        "date_max": part["ordered_on"].max(),
        "bad_dates": part["ordered_on"].isna().sum(),
    })

file_summary = pd.DataFrame(file_summary)
file_summary

In [ ]:
df = pd.concat(frames, ignore_index=True)
del frames

print(f"Rows: {len(df):,}")
print(f"Columns including provenance column: {df.shape[1]}")
print(f"Date range: {df['ordered_on'].min().date()} to {df['ordered_on'].max().date()}")

For the snapshot used to develop this project, the portal export happened to arrive as three CSV parts totaling **1,402,652 rows** across **2026-01-01 through 2026-07-31**.

Your file-level row counts may look different if the Iowa portal packages the export differently. That is okay. What matters is that:

- every source file has the same schema,
- the combined records cover the intended analysis window,
- no source chunks introduce unintended duplicate rows, and
- the combined audit results reconcile to the project snapshot, subject to any later corrections made by the public data publisher.


## 3. Duplicate audit

We distinguish two concepts:

- **Exact duplicate record:** every source field is identical. These are candidates for removal.
- **Repeated invoice + item:** the same item appears more than once on the same invoice. These are **not automatically duplicates** because quantities and dollar values can differ.


In [ ]:
SOURCE_COLUMNS = [c for c in df.columns if c != "_source_file"]

exact_duplicate_mask = df.duplicated(subset=SOURCE_COLUMNS, keep=False)
exact_duplicate_extra = df.duplicated(subset=SOURCE_COLUMNS, keep="first")

print(f"Rows involved in exact duplicate groups: {exact_duplicate_mask.sum():,}")
print(f"Extra exact duplicate rows: {exact_duplicate_extra.sum():,}")

df.loc[exact_duplicate_mask].sort_values(["invoice_id", "item_no"]).head(20)

In [ ]:
invoice_item_repeat_mask = df.duplicated(
    subset=["invoice_id", "item_no"],
    keep=False,
)

print(f"Rows involved in repeated invoice-item combinations: {invoice_item_repeat_mask.sum():,}")

df.loc[
    invoice_item_repeat_mask,
    ["invoice_id", "ordered_on", "store_no", "item_no", "im_desc", "sales_bottles", "sales_dollars"]
].sort_values(["invoice_id", "item_no"]).head(20)

**Decision:** remove only the 13 extra **exact** duplicates. Do **not** deduplicate on `invoice_id + item_no`; legitimate repeated lines exist with different quantities.

In [ ]:
df = df.drop_duplicates(subset=SOURCE_COLUMNS, keep="first").copy()
assert len(df) == 1_402_639
print(f"Rows after exact deduplication: {len(df):,}")

## 4. Missing-data audit

In [ ]:
missing = (
    df[SOURCE_COLUMNS]
    .isna()
    .sum()
    .rename("missing_rows")
    .to_frame()
)
missing["missing_pct"] = missing["missing_rows"] / len(df) * 100
missing = missing.sort_values("missing_rows", ascending=False)
missing.head(15)

The meaningful missingness is concentrated in two places:

- **Store geography:** 680 raw rows are missing address/city/ZIP/county. Twenty-one belong to store `4427` and can be filled from other rows for the **same store number**. The remaining 659 belong to store `010206` (`HAWKEYE SMOKE SHOP / FAIRFIELD`). The official Iowa Liquor Stores registry identifies store `010206` as **53 N COURT ST, FAIRFIELD**. Other Fairfield records in this snapshot consistently use ZIP `52556`, county FIPS `19101`, and county `JEFFERSON`.
- **Category:** 25 rows are missing both category code and category name, all for item `921635` (`TEMPLETON USA 250 STRAIGHT BOURBON WHISKEY`) from vendor `255` (`INFINIUM SPIRITS`). As checked in September 2026, Iowa's current inventory confirms the product identity but does not publish a category for this special-order item. Within this Iowa sales snapshot, `1011200` consistently maps to `STRAIGHT BOURBON WHISKIES`, and another Templeton bourbon item from the same vendor (`21612`, `TEMPLETON BOURBON`) is assigned to that category. We therefore treat `1011200 / STRAIGHT BOURBON WHISKIES` as a **documented researched inference**, not as a category directly supplied for item `921635` by the source.

External references used for the researched enrichment:

- Iowa Liquor Stores registry: https://data.iowa.gov/api/views/ykb6-ywnd/rows.pdf
- Iowa Inventory & Deliveries snapshot: https://shop.iowaabd.com/snapshot/inventory?order=asc&sort=category


In [ ]:
LOCATION_COLS = [
    "store_address",
    "store_city",
    "store_zip_code",
    "county_fips_code",
    "county_name",
]

before_location_missing = df["store_address"].isna().sum()

# First use only internal evidence: fill missing geography from other rows for the same store number.
location_lookup = (
    df.dropna(subset=["store_address"])
      .sort_values("ordered_on")
      .groupby("store_no")[LOCATION_COLS]
      .last()
)

for col in LOCATION_COLS:
    df[col] = df[col].fillna(df["store_no"].map(location_lookup[col]))

after_same_store_fill = df["store_address"].isna().sum()

# Store 010206 has no known geography inside this snapshot.
# The Iowa Liquor Stores registry identifies it as Hawkeye Smoke Shop, 53 N Court St, Fairfield.
# ZIP/county values below match other Fairfield rows in the Iowa sales data.
verified_010206_location = {
    "store_address": "53 N COURT ST",
    "store_city": "FAIRFIELD",
    "store_zip_code": "52556",
    "county_fips_code": "19101",
    "county_name": "JEFFERSON",
}

store_010206_mask = df["store_no"].eq("010206")
for col, value in verified_010206_location.items():
    df.loc[store_010206_mask, col] = df.loc[store_010206_mask, col].fillna(value)

after_verified_location_fill = df["store_address"].isna().sum()

print(f"Missing store geography before repair: {before_location_missing:,}")
print(f"Missing after same-store fill: {after_same_store_fill:,}")
print(f"Missing after verified 010206 enrichment: {after_verified_location_fill:,}")


In [ ]:
df.loc[
    df["store_address"].isna(),
    ["store_no", "store_name", *LOCATION_COLS]
].drop_duplicates()

### Category enrichment for item `921635`

The source leaves category fields blank for all 25 rows of `TEMPLETON USA 250 STRAIGHT BOURBON WHISKEY`. We do **not** classify it from the word “bourbon” alone. The enrichment is restricted to this exact item number and description, and is supported by the Iowa category taxonomy already present in the dataset plus the same-vendor Templeton bourbon comparison described above. Existing non-missing category values are never overwritten.


In [ ]:
before_category_missing = df["category_code"].isna().sum()

templeton_250_mask = (
    df["item_no"].eq("921635")
    & df["im_desc"].eq("TEMPLETON USA 250 STRAIGHT BOURBON WHISKEY")
)

# Fill only missing values for this specifically researched item.
df.loc[templeton_250_mask, "category_code"] = (
    df.loc[templeton_250_mask, "category_code"].fillna("1011200")
)
df.loc[templeton_250_mask, "category_name"] = (
    df.loc[templeton_250_mask, "category_name"].fillna("STRAIGHT BOURBON WHISKIES")
)

after_category_missing = df["category_code"].isna().sum()

print(f"Missing category rows before enrichment: {before_category_missing:,}")
print(f"Rows matched for item 921635: {templeton_250_mask.sum():,}")
print(f"Missing category rows after enrichment: {after_category_missing:,}")


**Decision:** retain all valid transactions and repair the concentrated missingness with traceable evidence. The 21 internally recoverable geography rows are filled from same-store history; the remaining store `010206` geography is enriched from the official Iowa store registry plus dataset-consistent Fairfield ZIP/county values. For item `921635`, the 25 missing category rows are filled as `1011200 / STRAIGHT BOURBON WHISKIES` using the documented researched inference above.

For the verified January–July 2026 snapshot, the expected result after these repairs is **0 rows with missing store geography** and **0 rows with missing category code/name**. The raw CSV files remain unchanged; the enrichment exists only in the processed analytical dataset.


## 5. Returns and negative transactions

Negative values are not automatically data errors. In this dataset they overwhelmingly represent return invoices (`RINV`). One negative row uses an `INV` prefix, so the transaction flag should be based on the numeric values rather than invoice text alone.

In [ ]:
df["is_return"] = (df["sales_bottles"] < 0) | (df["sales_dollars"] < 0)

print(f"Return / negative rows: {df['is_return'].sum():,}")
print(df.loc[df["is_return"], "invoice_id"].str.extract(r"^([A-Za-z]+)")[0].value_counts())

df.loc[
    df["is_return"],
    ["invoice_id", "ordered_on", "store_no", "item_no", "im_desc", "sales_bottles", "sales_dollars"]
].head(10)

**Decision:** keep returns. Removing them would overstate net sales and erase a real business process. Later notebooks can calculate both gross sales and net sales when appropriate.

## 6. Numeric and logical checks

In [ ]:
NUMERIC_COLS = [
    "pack",
    "bottle_volume_ml",
    "state_bottle_cost",
    "state_bottle_retail",
    "sales_bottles",
    "sales_dollars",
    "sales_liters",
    "sales_gallons",
]

numeric_audit = pd.DataFrame({
    "min": df[NUMERIC_COLS].min(),
    "max": df[NUMERIC_COLS].max(),
    "zero_count": (df[NUMERIC_COLS] == 0).sum(),
    "negative_count": (df[NUMERIC_COLS] < 0).sum(),
})
numeric_audit

One small anomaly is worth preserving as a quality flag: 42 rows for item `32232` have `state_bottle_cost == 0`. The rows still contain sales dollars, so they remain usable for revenue analysis but should not be used for margin calculations without further source validation.

In [ ]:
df["has_zero_source_cost"] = df["state_bottle_cost"].eq(0)

df.loc[
    df["has_zero_source_cost"],
    ["ordered_on", "item_no", "im_desc", "state_bottle_cost", "state_bottle_retail", "sales_bottles", "sales_dollars"]
].head()

## 7. Detect the January 2026 precision issue

This is the most important audit finding.

For January, nearly every value in these fields is stored as a whole number:

- `state_bottle_cost`
- `state_bottle_retail`
- `sales_liters`
- `sales_gallons`

From February onward, decimal precision returns. `sales_dollars`, `sales_bottles`, and `bottle_volume_ml` remain usable, which gives us a way to reconstruct precise volume and an effective unit sale price.

In [ ]:
PRECISION_COLS = [
    "state_bottle_cost",
    "state_bottle_retail",
    "sales_liters",
    "sales_gallons",
]

precision_check = []
for month, group in df.groupby(df["ordered_on"].dt.to_period("M")):
    row = {"month": str(month), "rows": len(group)}
    for col in PRECISION_COLS:
        s = group[col].dropna().astype(float)
        row[f"{col}_integer_share"] = np.isclose(s, np.round(s)).mean()
    precision_check.append(row)

precision_check = pd.DataFrame(precision_check)
precision_check

You should see a sharp structural break between January and February, not random noise. That means we should treat this as a **source precision problem**, not as thousands of independent bad transactions.

We do **not** overwrite the raw source columns. Instead, we add trustworthy derived fields and a flag so future notebooks cannot accidentally use January cost/retail values as if they had full precision.

In [ ]:
JAN_END = pd.Timestamp("2026-02-01")
df["source_price_precision_ok"] = df["ordered_on"] >= JAN_END

# Precise volume can be reconstructed exactly from bottle count and bottle size.
df["volume_liters_calc"] = df["sales_bottles"] * df["bottle_volume_ml"] / 1000
df["volume_gallons_calc"] = df["volume_liters_calc"] / 3.785411784

# Revenue per bottle is recoverable from transaction totals.
df["effective_unit_price"] = np.where(
    df["sales_bottles"].ne(0),
    df["sales_dollars"] / df["sales_bottles"],
    np.nan,
)

# Safe downstream versions of source price/cost fields.
# January is set to missing because its cents precision was lost in the source snapshot.
df["state_bottle_cost_usable"] = df["state_bottle_cost"].where(df["source_price_precision_ok"])
df["state_bottle_retail_usable"] = df["state_bottle_retail"].where(df["source_price_precision_ok"])

df[[
    "ordered_on", "sales_bottles", "bottle_volume_ml",
    "sales_liters", "volume_liters_calc",
    "state_bottle_retail", "effective_unit_price",
    "source_price_precision_ok"
]].head()

### Why not reconstruct cost?

Volume is mechanically determined by bottle count × bottle size, so recalculating it is defensible. Transaction revenue also lets us compute an **effective unit sale price**.

January bottle **cost**, however, cannot be recovered exactly from the transaction fields without imposing an external pricing assumption. We therefore leave January cost unavailable for downstream margin analysis instead of fabricating precision.

## 8. Add analysis-friendly time fields

In [ ]:
df["year"] = df["ordered_on"].dt.year.astype("int16")
df["month"] = df["ordered_on"].dt.month.astype("int8")
df["month_name"] = df["ordered_on"].dt.month_name().astype("category")
df["week"] = df["ordered_on"].dt.to_period("W-SUN").dt.start_time
df["day_of_week"] = df["ordered_on"].dt.day_name().astype("category")

df[["ordered_on", "year", "month", "month_name", "week", "day_of_week"]].head()

## 9. Final quality summary

In [ ]:
quality_summary = pd.Series({
    "clean_rows": len(df),
    "date_min": df["ordered_on"].min().date(),
    "date_max": df["ordered_on"].max().date(),
    "unique_invoices": df["invoice_id"].nunique(),
    "unique_stores": df["store_no"].nunique(),
    "unique_items": df["item_no"].nunique(),
    "known_categories": df["category_code"].nunique(),
    "known_counties": df["county_fips_code"].nunique(),
    "return_rows": int(df["is_return"].sum()),
    "remaining_missing_geography_rows": int(df["store_address"].isna().sum()),
    "missing_category_rows": int(df["category_code"].isna().sum()),
    "zero_source_cost_rows": int(df["has_zero_source_cost"].sum()),
    "jan_precision_flagged_rows": int((~df["source_price_precision_ok"]).sum()),
})
quality_summary

Expected headline values after cleaning:

- **1,402,639 rows** after exact deduplication
- **72,456 invoices**
- **2,183 stores**
- **4,276 items**
- **44 known category codes**
- **99 known Iowa counties**
- **1,088 return/negative rows**
- **0 rows** still lacking store geography after same-store and verified external enrichment
- **0 rows** still lacking category information after the documented item `921635` enrichment

The January precision finding is retained as a first-class data quality constraint rather than hidden by the cleaning process.


## 10. Save the processed dataset

Parquet is preferable to CSV for the processed layer because it preserves data types, is smaller, and reloads much faster.

Install the Parquet engine once in your project environment if needed:

```bash
python -m pip install pyarrow
```


In [ ]:
OUTPUT_PATH = PROCESSED_DIR / "iowa_liquor_sales_2026_clean.parquet"
AUDIT_PATH = PROCESSED_DIR / "iowa_liquor_sales_2026_audit_summary.csv"

# The provenance column is useful during audit but not required in the analytical table.
df.drop(columns=["_source_file"]).to_parquet(OUTPUT_PATH, index=False)
quality_summary.rename("value").to_csv(AUDIT_PATH, header=True)

print(f"Saved: {OUTPUT_PATH}")
print(f"Saved: {AUDIT_PATH}")

## Cleaning decisions for the project README

| Issue | Finding | Decision |
|---|---|---|
| Source-file schema | All downloaded files match | Concatenate |
| Cross-file overlap | No unintended exact overlap in the verified snapshot | Keep all valid source records |
| Exact duplicates | 13 extra rows | Remove exact duplicates only |
| Repeated invoice-item keys | Legitimate rows with different quantities | Keep |
| Negative sales | 1,088 rows; mostly `RINV` | Keep and flag as returns |
| Missing store geography | 680 raw rows | Fill 21 from same-store history; enrich store `010206` from the Iowa store registry and dataset-consistent Fairfield geography; 0 remain |
| Missing category | 25 rows for item `921635` | Fill `1011200 / STRAIGHT BOURBON WHISKIES` as a documented researched inference; 0 remain |
| Zero source cost | 42 rows for one item | Keep; exclude from cost/margin use |
| January numeric precision | Cost, retail, liters, gallons mostly lose decimals | Flag Jan source prices; recompute volume; use effective unit price |

### Downstream metric policy

**Safe for Jan–Jul:** `sales_dollars`, `sales_bottles`, derived volume, invoice/order behavior, product/category/store mix, effective unit price.

**Use only Feb–Jul unless independently validated:** source `state_bottle_cost` and `state_bottle_retail` for precise cost/price analysis.


## Next notebook

The data supports a stronger portfolio project than a hypothetical A/B test. The recommended direction is:

### **Store Segmentation + Assortment Opportunity Analysis**

Use transaction history to characterize Iowa liquor retailers by scale, order cadence, category mix, SKU breadth, premium mix, growth, and return behavior. Then identify products or categories that under-index within otherwise similar stores.

That gives the project a real decision question:

> **Which stores should a distributor prioritize, and what should it recommend to each store segment to increase sales?**

A later extension can add invoice-level product affinity / market-basket analysis to support cross-sell recommendations.